In [1]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

DATA_DIR = "bpi_mca_dataset"

train_traces = json.load(open(f"{DATA_DIR}/X_train.json"))
test_traces  = json.load(open(f"{DATA_DIR}/X_test.json"))

df = pd.read_csv(f"{DATA_DIR}/y_test.csv")
test_labels = df["label"].values

print("Train:", len(train_traces))
print("Test:", len(test_traces))

Train: 9963
Test: 9964


In [2]:
all_activities = set(a for t in train_traces for a in t)
V = len(all_activities)

In [3]:
from collections import defaultdict

n = 3  # trigram

ngram_counts = defaultdict(int)
context_counts = defaultdict(int)

for trace in train_traces:
    padded = ["START"] * (n - 1) + trace

    for i in range(len(trace)):
        context = tuple(padded[i:i+n-1])
        next_act = padded[i+n-1]

        ngram_counts[(context, next_act)] += 1
        context_counts[context] += 1

print("N-gram model built.")

N-gram model built.


In [4]:
def compute_ngram_score(trace):
    padded = ["START"] * (n - 1) + trace
    score = 0

    for i in range(len(trace)):
        context = tuple(padded[i:i+n-1])
        next_act = padded[i+n-1]

        prob = (ngram_counts[(context, next_act)] + 1) / (
            context_counts[context] + V
        )

        score += -np.log(prob)

    return score / len(trace)

In [5]:
train_scores = np.array([
    compute_ngram_score(t) for t in train_traces
])

test_scores = np.array([
    compute_ngram_score(t) for t in test_traces
])

In [6]:
threshold = np.percentile(train_scores, 70)

preds = (test_scores >= threshold).astype(int)

In [7]:
print("Confusion Matrix:")
print(confusion_matrix(test_labels, preds))

print("\nClassification Report:")
print(classification_report(test_labels, preds))

auc = roc_auc_score(test_labels, test_scores)
print("ROC-AUC:", auc)

Confusion Matrix:
[[1693  798]
 [1000 6473]]

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.68      0.65      2491
           1       0.89      0.87      0.88      7473

    accuracy                           0.82      9964
   macro avg       0.76      0.77      0.77      9964
weighted avg       0.82      0.82      0.82      9964

ROC-AUC: 0.9165970597321774
